# Day 5 — Autograd (Automatic Differentiation)

## 1. Learning Objectives
- Understand PyTorch's Computational Graph.
- Use `requires_grad=True` to track operations.
- Compute gradients using `backward()`.
- Understand gradient accumulation.
- Learn how to stop tracking gradients using `torch.no_grad()` and `.detach()`.

## 2. Prerequisites
- Basic calculus (understanding what a derivative/gradient is).
- Tensor operations (Day 3).

In [ ]:
import torch

## 3. Concept Explanation
**Autograd** is PyTorch's automatic differentiation engine. It powers neural network training.

When you create a tensor with `requires_grad=True`, PyTorch starts tracking every operation applied to that tensor. It builds a **Computational Graph** in the background. When you calculate a final output (like a loss) and call `.backward()`, PyTorch traverses this graph backwards to calculate the gradient of the output with respect to the input tensor.

## 4. Why This Matters
Without Autograd, you would have to calculate the derivatives of massive neural networks by hand using the chain rule. Autograd does this automatically, exact, and extremely fast.

## 6. Mathematical Foundation
Let $y = 3x^2 + 2x$. 
The derivative with respect to $x$ is: $\frac{dy}{dx} = 6x + 2$.
If $x = 2$, then $y = 12 + 4 = 16$. 
And the gradient at $x=2$ is $\frac{dy}{dx} = 6(2) + 2 = 14$.

Let's see PyTorch do this automatically.

In [ ]:
# 8. Simple Example
x = torch.tensor(2.0, requires_grad=True)

# Forward pass (Building the computational graph)
y = 3 * x**2 + 2 * x

print("Output y:", y.item())

# Backward pass (Computing gradients)
y.backward()

# The gradient dy/dx is stored in x.grad
print("Gradient dy/dx at x=2:", x.grad.item())

## 9. Code Walkthrough
1. We define `x` as a float tensor and set `requires_grad=True`. 
2. We perform math to get `y`. PyTorch secretly builds a graph linking `y` to `x`.
3. `y.backward()` triggers the chain rule.
4. `x.grad` contains the answer (14.0).

## 10. Experiment: Gradient Accumulation
A very important quirk in PyTorch: **gradients accumulate** (add up) by default if you call `.backward()` multiple times. You must explicitly zero them out in a real training loop.

In [ ]:
w = torch.tensor(1.0, requires_grad=True)

for _ in range(3):
    loss = w * 2
    loss.backward()
    print(f"Gradient after backward: {w.grad.item()}")
    
# How to zero out gradients:
w.grad.zero_()
print(f"Gradient after zeroing: {w.grad.item()}")

## 11. Practice Exercise 1: torch.no_grad()
When we are just evaluating a model (not training it), we don't want PyTorch to build a computational graph because it wastes memory and time. We use the context manager `torch.no_grad()`.

**Task:** Create a tensor `a` with `requires_grad=True`. Perform `b = a * 2` inside a `torch.no_grad()` block. Print `b.requires_grad` to verify it is False.

In [ ]:
# Write your code here

In [ ]:
# SOLUTION
a = torch.tensor([1.0, 2.0], requires_grad=True)

with torch.no_grad():
    b = a * 2
    
print("Does b require grad?", b.requires_grad) # Should be False

## 13. Debugging Challenge
Why does the following code fail?

In [ ]:
z = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
output = z * 2

# output.backward() # Uncomment to see error

**Solution:** You can only call `.backward()` implicitly on a **scalar** (a 0-dimensional tensor with a single value). `output` is a vector. PyTorch doesn't know how to compute the gradient of a vector w.r.t a vector implicitly. You usually aggregate the output into a scalar first (like calculating the mean loss: `output.mean().backward()`).

## 17. Interview Questions
1. **What is a Computational Graph?**
   *Answer*: A directed graph where nodes are operations and edges are tensors. It tracks the sequence of operations applied to inputs to calculate outputs, enabling reverse-mode automatic differentiation.
2. **What does `.detach()` do?**
   *Answer*: It creates a new tensor that shares the same data with the original tensor, but is completely disconnected from the computational graph. It will never require gradients.

## 19. Day Summary
- `requires_grad=True` starts the computational graph tracking.
- `.backward()` computes the gradients and stores them in `.grad`.
- Gradients **accumulate**. You must zero them out.
- Use `with torch.no_grad():` during inference/evaluation to save memory.